In [18]:
%pip install -r ../requirements.txt requests 

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.2.1 -> 26.3
[notice] To update, run: python.exe -m pip install --upgrade pip


In [22]:
# Hjælpefunktionen - viser intet output, når den kører. Den definerer bare funktionen
import requests
import pandas as pd

# Politiets/CBS' tabel med månedlige forbrydelser pr. kvarter
URL = URL = "https://dataderden.cbs.nl/ODataFeed/OData/47022NED/TypedDataSet"

def download(filter_text):
    """Henter alle rækker, der matcher filteret. API'et giver max 10.000 rækker ad gangen så vi henter side for side, indtil der ikke kommer flere"""
    rows, skip = [], 0
    while True:
        params = {"$filter": filter_text, "$format": "json", "$top": 10000, "$skip": skip}
        response = requests.get(URL, params=params, timeout=120)
        response.raise_for_status()
        page = response.json()["value"]
        if not page:
            break
        rows += page
        skip += 10000
        print(f"{len(rows):,} rækker hentet")
    return pd.DataFrame(rows)

In [23]:
print(URL)

https://dataderden.cbs.nl/ODataFeed/OData/47022NED/TypedDataSet


In [24]:
# '1.1.1 ' = boligindbrud (mellemrummet til sidst skal med)
# 'BU0363' = alle kvarterer i Amsterdam
test = download("SoortMisdrijf eq '1.1.1 ' and substringof('BU0363', WijkenEnBuurten)")
print(test.shape)
test.head()

10,000 rækker hentet
20,000 rækker hentet
30,000 rækker hentet
40,000 rækker hentet
50,000 rækker hentet
60,000 rækker hentet
70,000 rækker hentet
80,000 rækker hentet
90,000 rækker hentet
91,509 rækker hentet
(91509, 5)


,ID,SoortMisdrijf,WijkenEnBuurten,Perioden,GeregistreerdeMisdrijven_1
0,3481590,1.1.1,BU0363AA01,2012MM01,0
1,3481591,1.1.1,BU0363AA01,2012MM02,0
2,3481592,1.1.1,BU0363AA01,2012MM03,0
3,3481593,1.1.1,BU0363AA01,2012MM04,1
4,3481594,1.1.1,BU0363AA01,2012MM05,0
